### Preparations

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [22]:
p_base = "../Dataset/ml-100k/u1.base"
p_test = "../Dataset/ml-100k/u1.test"
f_base = pd.read_csv(p_base,sep='\t',names=['user_id','item_id','rating','timestamp'])
f_test = pd.read_csv(p_test,sep='\t',names=['user_id','item_id','rating','timestamp'])
print(f_base)
print(f_test)

       user_id  item_id  rating  timestamp
0            1        1       5  874965758
1            1        2       3  876893171
2            1        3       4  878542960
3            1        4       3  876893119
4            1        5       3  889751712
...        ...      ...     ...        ...
79995      943     1067       2  875501756
79996      943     1074       4  888640250
79997      943     1188       3  888640250
79998      943     1228       3  888640275
79999      943     1330       3  888692465

[80000 rows x 4 columns]
       user_id  item_id  rating  timestamp
0            1        6       5  887431973
1            1       10       3  875693118
2            1       12       5  878542960
3            1       14       5  874965706
4            1       17       3  875073198
...        ...      ...     ...        ...
19995      458      648       4  886395899
19996      458     1101       4  886397931
19997      459      934       3  879563639
19998      460       10     

In [23]:
user_ratings = {} # {user_id:[rating1,rating2,...]}
for (u,i,r,ts) in f_base.values:
    if u not in user_ratings:
        user_ratings[u] = []
    user_ratings[u].append(r)

item_ratings = {} # {item_id:[rating1,rating2,...]}
for (u,i,r,ts) in f_base.values:
    if i not in item_ratings:
        item_ratings[i] = []
    item_ratings[i].append(r)

In [24]:
user_mean = {} # {user_id:[um1,um2,...]}
item_mean = {} # {item_id:[im1,im2,...]}

for u,ratings in user_ratings.items():
    user_mean[u] = sum(ratings) / len(ratings)

for i,ratings in item_ratings.items():
    item_mean[i] = sum(ratings) / len(ratings)

In [25]:
user_items = {} # {user_id:{item_id:[rating]}}
for (u,i,r,ts) in f_base.values:
    if u not in user_items:
        user_items[u] = {}
    user_items[u][i] = r

item_users = {} # {item_id:{user_id:[rating]}}
for (u,i,r,ts) in f_base.values:
    if i not in item_users:
        item_users[i] = {}
    item_users[i][u] = r

### UCF

In [26]:
# PCC
S_wu = {}
users = list(user_items.keys())
for idx_w in range(len(users)):
    w = users[idx_w]
    items_w = user_items[w]
    r_bar_w = user_mean[w]
    for idx_u in range(idx_w + 1,len(users)):
        u = users[idx_u]
        items_u = user_items[u]
        r_bar_u = user_mean[u]
        common_items = []
        for item in items_w:
            if item in items_u:
                common_items.append(item)
        if len(common_items) < 2:
            continue

        numerator = 0.0
        for item in common_items:
            numerator += (items_u[item] - r_bar_u) * (items_w[item] - r_bar_w)
        
        denom_u = 0.0
        for item in common_items:
            denom_u += pow((items_u[item] - r_bar_u),2)

        denom_w = 0.0
        for item in common_items:
            denom_w += pow((items_w[item] - r_bar_w),2)

        denominator = math.sqrt(denom_u) * math.sqrt(denom_w)

        if denominator == 0.0:
            continue

        S_wu[(w,u)] = numerator / denominator


In [27]:
user_sims = {}
for (w,u), sim in S_wu.items():
    
    if w not in user_sims:
        user_sims[w] = {}

    if u not in user_sims:
        user_sims[u] = {}

    user_sims[w][u] = sim
    user_sims[u][w] = sim

In [28]:
K = 50
preds_ucf = [] # [pred_rating1,pred_rating2,...]

for u,i,r,ts in f_test.values:
    # 1). 先找出除自己外评过物品j的所有用户集合candidates。
    candidates = []
    for w in user_items:
        if i in user_items[w] and w != u:
            candidates.append(w)
    # candidates = [w for w in user_items if i in user_items[w] and w != u]

    # 2). 在候选用户集合中，按与用户u的相似度取top-K (只留sim > 0)
    scored = []
    for w in candidates:
        s  = user_sims[u].get(w,0.0)
        scored.append((w,s))
    filtered = []
    for w,s in scored:
        if s > 0:
            filtered.append((w,s))
    scored = filtered

    # scored = [(w,user_sims[u].get(w,0.0)) for w in candidates]
    # scored = [(w,s) for (w,s) in scored if s > 0]

    scored.sort(key = lambda x : x[1],reverse=True) # 降序
    # scored.sort(key = lambda x : -x[1]) # 降序

    top_K = scored[:K]

    # 3). 预测: 此时top_K集合里的用户一定都评价过物品j
    numerator = 0.0
    denominator = 0.0
    for w,sim in top_K:
        numerator += sim * (user_items[w][i] - user_mean[w])
        denominator += abs(sim)

    if denominator > 0 :
        r_hat = user_mean[u] + numerator / denominator
    else:
        r_hat = user_mean[u]

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_ucf.append(r_hat)


### Metric

In [29]:
sse = 0.0
sae = 0.0

for idx,(u,i,r,ts) in enumerate(f_test.values):
    err = r - preds_ucf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"UCF RMSE = {(sse / len(preds_ucf)) **0.5:.4f}")
print(f"UCF MAE = {sae / len(preds_ucf) :.4f}")

UCF RMSE = 0.9544
UCF MAE = 0.7467


### ICF

In [30]:
# ACS
S_kj = {}
items = list(item_users.keys())

for idx_k in range(len(items)):
    k = items[idx_k]
    users_k = item_users[k]
    for idx_j in range(idx_k + 1,len(items)):
        j = items[idx_j]
        users_j = item_users[j]
        common_users = []
        for user in users_k:
            if user in users_j:
                common_users.append(user)
        if len(common_users) < 2:
            continue

        numerator = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            numerator += (users_k[user] - r_bar_user) * (users_j[user] - r_bar_user)

        denom_k = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_k += (users_k[user] - r_bar_user) ** 2

        denom_j = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_j += (users_j[user] - r_bar_user) ** 2

        denominator = math.sqrt(denom_k) * math.sqrt(denom_j)
        if denominator == 0.0:
            continue

        S_kj[(k,j)] = numerator / denominator


In [31]:
item_sims = {}
for (k,j), sim in S_kj.items():
    if k not in item_sims:
        item_sims[k] = {}
    if j not in item_sims:
        item_sims[j] = {}
    item_sims[k][j] = sim
    item_sims[j][k] = sim
    

In [32]:
K = 50
preds_icf = []
for u,i,r,ts in f_test.values:
    # 1) 候选物品 = 用户u评过的所有物品(排除目标物品j自己)
    candidates = []
    for k in item_users:
        if u in item_users[k] and i != k:
            candidates.append(k)
    # candidates = [k for k in item_users if u in item_users[k] and i != k]

    # 2) 在候选集中按与物品j的相似度取top-K只保留sim>0
    scored = [(k,item_sims.get(i,{}).get(k,0.0)) for k in candidates]
    scored = [(k,s) for (k,s) in scored if s > 0]

    scored.sort(key = lambda x:x[1],reverse=True) # 降序
    # scored.sort(key = lambda x:-x[1])

    top_K = scored[:K]

    # 3) 预测
    numerator = 0.0
    denominator = 0.0
    for k,sim in top_K:
        numerator += sim * item_users[k][u]
        denominator += abs(sim)

    if denominator > 0:
        r_hat = numerator / denominator
    else:
        r_hat = item_mean.get(i,user_mean[u])

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_icf.append(r_hat)


### Metric

In [33]:
sse = 0.0
sae = 0.0
for idx,(u,i,r,ts) in enumerate(f_test.values):
    err = r - preds_icf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"RMSE = {(sse / len(preds_icf)) ** 0.5:.4f}")
print(f"MAE = {sae / len(preds_icf):.4f}")

RMSE = 0.9845
MAE = 0.7753


### Hybrid_CF

In [34]:
preds_hybrid = []
lambda_ucf = 0.5
for r_ucf,r_icf in zip(preds_ucf,preds_icf):
    r_hybrid  = lambda_ucf * r_ucf + (1 - lambda_ucf) * r_icf
    preds_hybrid.append(r_hybrid)

### Metric

In [35]:
sse = 0.0
sae = 0.0
for idx,(u,i,r,ts) in enumerate(f_test.values):
    err = r - preds_hybrid[idx]
    sse += err ** 2
    sae += abs(err)

print(f"RMSE = {(sse / len(preds_hybrid))**0.5:.4f}")
print(f"MAE = {sae / len(preds_hybrid):.4f}")

RMSE = 0.9526
MAE = 0.7508
